In [4]:
import datetime
import sys
import pandas as pd
import datetime

In [5]:
end_date = pd.to_datetime(datetime.datetime.today().date())
start_date = pd.to_datetime(datetime.datetime.today().date() - datetime.timedelta(days=14))

year = datetime.datetime.today().year
month = datetime.datetime.today().month
day = datetime.datetime.today().day
today = year*10000 + month*100 + day

print(f'今日是：{today}')
print(f'start_date：{start_date}')
print(f'end_date：{end_date}')

今日是：20260529
start_date：2026-05-15 00:00:00
end_date：2026-05-29 00:00:00


In [6]:
mass =  pd.read_excel('./eps3_mass.xlsx')
spare =  pd.read_excel('./eps3_spare.xlsx')

# spare_effected = pd.read_excel('./spare_effected.xlsx')
# spare_effected.to_excel('./spare_effected_copy.xlsx',index=False) # spare_effected_copy.xlsx仅用于一次保存，后续可删除

c:\Users\caicw\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\caicw\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


### 1、数据清洗：
1.已生效价格通知书号，2.生效日期/失效日期 只提取日期并转换成日期格式(后续需要比较)

In [7]:
mass = mass[mass['状态'].isin(['已发布','已确认','已审批'])]
spare = spare[spare['通知书状态']=='已审批']

mass['生效日期'] = pd.to_datetime(mass['生效日期'].str[:10])
spare['生效日期'] = pd.to_datetime(spare['生效日期'].str[:10])
mass['失效日期'] = pd.to_datetime(mass['失效日期'].str[:10])
spare['失效日期'] = pd.to_datetime(spare['失效日期'].str[:10])
mass['审批通过时间'] = pd.to_datetime(mass['审批通过时间'].str[:10])
spare['审批通过时间'] = pd.to_datetime(spare['审批通过时间'].str[:10])

In [8]:
usefull_mass_column = ['价格通知书号','供应商编码','零件号','零件裸价','生效日期','失效日期','地区','创建时间','创建人']
usefull_spare_column = ['价格通知书号','供应商编码','供应商名称','备件号','备件名称','量产件号','地区','备件裸价','备件物流费','备件包装费','生效日期','失效日期','创建时间','采购工程师姓名','创建人','审批通过时间']

In [9]:
# 如果想重新运转，可以从此行开始
df_mass_useful = mass[usefull_mass_column]
df_spare_useful = spare[usefull_spare_column]

### 3、处理spare数据
1.导出本期的备件通知书号(20251212起，使用字段"审批通过时间"),
2.删除无对应量产件号的备件,
3.对全部新增的备件号的有效期扩展成每月(新增行)

In [10]:
# 找到当期新增生效备件
df_spare_current = df_spare_useful[(df_spare_useful['审批通过时间']<=end_date)&(df_spare_useful['审批通过时间']>=start_date)]

print(f'本期新增生效备件零件数量为{df_spare_current['备件号'].count()}')

if df_spare_current['备件号'].count() == 0:
    exit()

本期新增生效备件零件数量为1277


In [11]:
# 删除无对应量产件号的备件号
df_spare_current_with_massnumber = df_spare_current.dropna(subset=['量产件号'])
df_spare_current_with_massnumber.shape

(1222, 16)

In [12]:
# 本期无新增备件或新增备件无对应量产件，停止
if df_spare_current_with_massnumber.shape[0] == 0:
    print("本期无新增备件或新增备件无对应量产件")
    exit(0)

else:
# 如本期有新增
# 对全部新增的备件号的有效期扩展成每月(新增行)
    def expland_monthly(df):
        date_months = pd.date_range(
            start=df['生效日期'],
            end=df['失效日期'],
            freq='ME'
        )
        return date_months.to_period('M')

    months = df_spare_current_with_massnumber.apply(expland_monthly,axis=1)

    df_spare_current_with_massnumber.loc[:,'month'] = months

    df_spare_current_with_massnumber = (
        df_spare_current_with_massnumber.explode('month')
        .sort_values(by=['备件号','month','创建时间'],ascending=False)
        .drop_duplicates(subset=['备件号','供应商编码','month'],keep='first')
    )
    df_spare_current_with_massnumber.head(2)

C:\Users\caicw\AppData\Local\Temp\ipykernel_13400\1183966702.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_spare_current_with_massnumber.loc[:,'month'] = months


### 4、处理mass输机
1.匹配到新的备件号对应的量产件号(同一家供应商)的全部价格通知书(转换数据类型，新增‘供应商-零件号’列)

2.将量产零件号生效日期和失效日期完全相同的价格通知书只保留最新输机的通知书

3.将保留下来的价格通知书有效期按照每月拆分

In [13]:
# 匹配到新的备件号对应的量产件号(同一家供应商)的全部价格通知书(转换数据类型，新增‘供应商-零件号’列)
df_spare_current_with_massnumber.loc[:,'供应商编码'] = df_spare_current_with_massnumber['供应商编码'].astype('str')
df_spare_current_with_massnumber.loc[:,'量产件号'] = df_spare_current_with_massnumber['量产件号'].astype('str')
df_spare_current_with_massnumber['供应商-零件号'] = df_spare_current_with_massnumber['供应商编码'] + df_spare_current_with_massnumber['量产件号']

df_mass_useful.loc[:,'供应商编码'] = df_mass_useful['供应商编码'].astype('str')
df_mass_useful.loc[:,'零件号'] = df_mass_useful['零件号'].astype('str')
df_mass_useful['供应商-零件号'] = df_mass_useful['供应商编码'] + df_mass_useful['零件号']


C:\Users\caicw\AppData\Local\Temp\ipykernel_13400\648012068.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_mass_useful['供应商-零件号'] = df_mass_useful['供应商编码'] + df_mass_useful['零件号']


In [14]:
# 将量产零件号生效日期和失效日期完全相同的价格通知书只保留最新输机的通知书
df_mass_in_spare_current = df_mass_useful[df_mass_useful['供应商-零件号'].isin(df_spare_current_with_massnumber['供应商-零件号'].unique())]
df_mass_in_spare_current = df_mass_in_spare_current.dropna(subset=['生效日期','失效日期'],how='any')
df_mass_in_spare_current = df_mass_in_spare_current.sort_values(by='创建时间',ascending=False)
df_mass_in_spare_current = df_mass_in_spare_current.drop_duplicates(subset=['供应商-零件号','生效日期','失效日期'],keep='first')
df_mass_in_spare_current.head(2)

,价格通知书号,供应商编码,零件号,零件裸价,生效日期,失效日期,地区,创建时间,创建人,供应商-零件号
101,CJ2605280021,100288,Z100161760,16975.44,2025-05-01,2025-05-31,Wuhan/,2026-05-28 10:33:52,彭琴,100288Z100161760
104,CJ2605280021,100288,Z100203360,7981.91,2025-05-01,2025-05-31,Wuhan/,2026-05-28 10:33:52,彭琴,100288Z100203360


##### 3.将保留下来的价格通知书有效期按照每月拆分

In [15]:
# 将保留下来的价格通知书有效期按照每月拆分
df_mass_in_spare_current.loc[:,'month'] = df_mass_in_spare_current.apply(expland_monthly,axis=1)
df_mass_in_spare_current = (
    df_mass_in_spare_current.explode('month')
    .sort_values(by=['供应商-零件号','month','创建时间'],ascending=False)
    .drop_duplicates(subset=['供应商-零件号','供应商编码','month'],keep='first')
)

### 5、备件履历与量产履历整合
1.merge()合并查询('供应商-零件号'、'month')

2.剔除未匹配到量产价格的备件

3.数据整理(重命名列，仅保留需求的列)


In [16]:
# merge()合并查询('供应商-零件号'、'month')
df_spare_with_mass_price = df_spare_current_with_massnumber.merge(
    df_mass_in_spare_current,
    on=['供应商-零件号','month'],
    how='left'
)

In [17]:
# 剔除未匹配到量产价格的备件
df_spare_with_mass_price = df_spare_with_mass_price[~df_spare_with_mass_price.isna().any(axis=1)]

In [18]:
df_spare_with_mass_price.columns

Index(['价格通知书号_x', '供应商编码_x', '供应商名称', '备件号', '备件名称', '量产件号', '地区_x', '备件裸价',
       '备件物流费', '备件包装费', '生效日期_x', '失效日期_x', '创建时间_x', '采购工程师姓名', '创建人_x',
       '审批通过时间', 'month', '供应商-零件号', '价格通知书号_y', '供应商编码_y', '零件号', '零件裸价',
       '生效日期_y', '失效日期_y', '地区_y', '创建时间_y', '创建人_y'],
      dtype='object')

In [19]:
# 数据整理(重命名列，仅保留需求的列)
df_spare_with_mass_price.rename(
    columns={'价格通知书号_x':'备件价格通知书','供应商编码_x':'供应商编码','生效日期_x':'备件生效日期','失效日期_x':'备件失效日期',
             '价格通知书号_y':'量产件价格通知书号','生效日期_y':'量产件生效日期','失效日期_y':'量产件失效日期','零件裸价':'量产件裸价',
             '创建人_y':'输机工程师','采购工程师姓名':'采购工程师'},
    inplace=True
)

columns_used_in_mass_with_spare = ['备件价格通知书','供应商编码','供应商名称','备件号','备件名称','备件裸价','备件生效日期','备件失效日期',
                                   '量产件号','量产件价格通知书号','量产件裸价','量产件生效日期','量产件失效日期','采购工程师','输机工程师','month']

df_spare_with_mass_price = df_spare_with_mass_price[columns_used_in_mass_with_spare]

### 6、找到同期裸价不一致项

In [20]:
df_spare_mass_price_different_month = df_spare_with_mass_price[(df_spare_with_mass_price['备件裸价'] > df_spare_with_mass_price['量产件裸价'])]

In [21]:
df_spare_mass_price_different = df_spare_mass_price_different_month.drop_duplicates(['备件号','备件价格通知书'],keep='first')
df_spare_mass_price_different = df_spare_mass_price_different.drop('month',axis=1)
df_spare_mass_price_different

,备件价格通知书,供应商编码,供应商名称,备件号,备件名称,备件裸价,备件生效日期,备件失效日期,量产件号,量产件价格通知书号,量产件裸价,量产件生效日期,量产件失效日期,采购工程师,输机工程师


In [22]:
df_spare_mass_price_different.to_excel(f'./备件高于同期量产件清单/spare_mass_price_difference_{today}.xlsx',index=False)